# ORION PG-19 Benchmark (Dense vs Window vs Sparse)

This notebook runs the **config-driven** PG-19 benchmark profile:

- profile: `configs/experiments/profiles/pg19_core_a100.yaml`
- variants: dense, window (`w64`), sparse (`w64,d8`, flex)
- sequence lengths: `2048, 4096, 8192`
- seeds: `123, 456, 789`

It downloads PG-19 during runtime (no dataset files in repo), caches under `data/pg19`, and summarizes quality/speed/VRAM.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_REF = os.environ.get("ORION_GIT_REF", "main")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/orion_long_context")
    REPO_ROOT = DRIVE_ROOT / "repo"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()


def _run(cmd: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({' '.join(cmd)}):\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc


def _is_git_repo(path: Path) -> bool:
    return (path / ".git").exists()


def _clone_repo(repo_url: str, repo_root: Path, retries: int = 3) -> None:
    last_error = "unknown"
    for attempt in range(1, retries + 1):
        proc = subprocess.run(["git", "clone", repo_url, str(repo_root)], text=True, capture_output=True)
        if proc.returncode == 0:
            return

        last_error = (proc.stderr or proc.stdout or "clone failed").strip()
        tail = last_error.splitlines()[-1] if last_error else "clone failed"
        print(f"Clone attempt {attempt}/{retries} failed: {tail}")

        if repo_root.exists() and not _is_git_repo(repo_root):
            shutil.rmtree(repo_root, ignore_errors=True)

        if attempt < retries:
            time.sleep(attempt)

    raise RuntimeError(f"Failed to clone repository after {retries} attempts: {last_error}")


def _sync_repo(repo_root: Path, ref: str) -> None:
    status = _run(["git", "status", "--porcelain"], cwd=repo_root)
    if status.stdout.strip():
        raise RuntimeError(
            "Repository has local changes in Drive clone. Commit/stash/remove them, or delete the repo folder and rerun."
        )

    _run(["git", "fetch", "--all", "--tags", "--prune"], cwd=repo_root)

    checkout = _run(["git", "checkout", ref], cwd=repo_root, check=False)
    if checkout.returncode != 0:
        _run(["git", "checkout", "-B", ref, f"origin/{ref}"], cwd=repo_root)

    branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_root).stdout.strip()
    if branch != "HEAD":
        _run(["git", "pull", "--ff-only", "origin", branch], cwd=repo_root)


if REPO_ROOT.exists() and not _is_git_repo(REPO_ROOT):
    if any(REPO_ROOT.iterdir()):
        backup = REPO_ROOT.with_name(f"{REPO_ROOT.name}_backup_{int(time.time())}")
        print(f"Found non-git directory at {REPO_ROOT}; moving to {backup}")
        REPO_ROOT.rename(backup)
    else:
        REPO_ROOT.rmdir()

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    _clone_repo(REPO_URL, REPO_ROOT)
else:
    print(f"Using existing repository at {REPO_ROOT}")

_sync_repo(REPO_ROOT, REPO_REF)
os.chdir(REPO_ROOT)

commit = _run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT).stdout.strip()
branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_ROOT).stdout.strip()

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"Checked out: {branch} @ {commit}")


In [ ]:
print(f"Python: {sys.version}")


def _pip_install(args: list[str], *, optional: bool = False) -> bool:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.returncode == 0:
        return True

    print(f"\n[pip failed] {' '.join(cmd)}")
    if proc.stdout.strip():
        print("--- pip stdout (tail) ---")
        print("\n".join(proc.stdout.splitlines()[-40:]))
    if proc.stderr.strip():
        print("--- pip stderr (tail) ---")
        print("\n".join(proc.stderr.splitlines()[-80:]))

    if optional:
        print("Continuing because dependency is optional.")
        return False

    raise RuntimeError("Required dependency installation failed.")


_pip_install(["-U", "pip"])
_pip_install(["-r", "requirements.txt"])
_pip_install(["-e", "."])
_pip_install(["datasets>=2.19"])
_pip_install(["pandas", "seaborn", "matplotlib"])

print("Dependencies installed.")


In [ ]:
from pathlib import Path

import seaborn as sns

from orion.experiments import (
    load_summary_df,
    paired_analysis_tables,
    plot_speedup_ratios,
    plot_vram_and_quality,
    prepare_analysis_columns,
    run_profile,
    select_winners,
)

sns.set_theme(style="whitegrid")

PROFILE = "pg19_core_a100"
OVERWRITE = False
VAL_PPL_TOLERANCE = 0.20

run_result = run_profile(PROFILE, overwrite=OVERWRITE)

print("Experiment ID:", run_result.experiment_id)
print("Summary CSV:", run_result.summary_csv)
print("Runs root:", run_result.runs_root)


In [ ]:
df_raw = load_summary_df(run_result.summary_csv)
df = prepare_analysis_columns(df_raw)
df_ok = df[df["status"] == "ok"].copy()

print("Successful trials:", len(df_ok), "/", len(df))
display(df_ok.head(20))

summary = (
    df_ok.groupby(["backend", "sparse_tag", "seq_len"], as_index=False)
    .agg(
        seeds=("seed", "nunique"),
        val_ppl_mean=("val_ppl", "mean"),
        val_ppl_std=("val_ppl", "std"),
        acc_mean=("final_acc_top1", "mean"),
        acc_std=("final_acc_top1", "std"),
        train_tok_s_mean=("train_tok_per_s", "mean"),
        train_tok_s_std=("train_tok_per_s", "std"),
        peak_vram_gb_mean=("peak_vram_gb", "mean"),
        peak_vram_gb_std=("peak_vram_gb", "std"),
    )
    .sort_values(["seq_len", "backend", "sparse_tag"])
)

print("Mean ± std summary:")
display(summary)


In [ ]:
paired_all, paired_focus, agg = paired_analysis_tables(df_ok)

print(f"Paired rows (all non-dense variants): {len(paired_all)}")
print(f"Paired rows (focused): {len(paired_focus)}")
display(agg.sort_values(["seq_len", "backend", "sparse_tag", "lr"]))

winners = select_winners(agg, val_ppl_tolerance=VAL_PPL_TOLERANCE)
print("Candidate wins over dense:")
display(winners)

plot_speedup_ratios(paired_focus)
plot_vram_and_quality(paired_focus)


## Notes

- PG-19 is downloaded at runtime via Hugging Face and token cache is stored under `data/pg19/cache_*`.
- Use `OVERWRITE=True` only when you want to rerun all trials from scratch.
- If your runtime disconnects, rerun the notebook with `OVERWRITE=False` to resume from completed runs.
